In [2]:
import pandas as pd
import numpy as np

def gerar_dataset_gatos(n=1000, seed=42):
    rng = np.random.default_rng(seed)

    sexos = np.array(["M", "F"])
    cores = np.array(["preto", "branco", "cinza", "laranja", "tigrado", "tricolor", "siamês"])
    pelagens = np.array(["curta", "média", "longa"])
    temperamentos = np.array(["calmo", "brincalhão", "arisco", "carinhoso", "caçador"])
    alimentacao = np.array(["ração seca", "ração úmida", "mista"])

    # ====== NOVO: id_tutor_hash (falso, sem PII) ======
    n_tutores = max(50, n // 8)
    tutores_hash = np.array([f"T{rng.integers(10**9, 10**10)}" for _ in range(n_tutores)])
    id_tutor_hash = rng.choice(tutores_hash, size=n, replace=True)

    # ====== NOVO: data_coleta (data sintética) ======
    inicio = np.datetime64("2025-01-01")
    fim = np.datetime64("2025-12-31")
    dias = (fim - inicio).astype(int)

    offsets = rng.integers(0, dias + 1, size=n)
    data_coleta = inicio + offsets.astype("timedelta64[D]")

    # ====== Idade (0 a 18) com distribuição mais realista ======
    idade_probs = np.array([0.06, 0.07, 0.08, 0.08, 0.07, 0.07, 0.06, 0.06, 0.06,
                            0.06, 0.05, 0.05, 0.04, 0.04, 0.03, 0.03, 0.02, 0.01, 0.01])

    # ✅ Correção: normaliza as probabilidades para somar 1
    idade_probs = idade_probs / idade_probs.sum()

    idade_anos = rng.choice(np.arange(0, 19), size=n, p=idade_probs)

    # Sexo
    sexo = rng.choice(sexos, size=n, p=[0.48, 0.52])

    # Peso (kg) baseado em sexo e idade (didático, não médico)
    base_peso = np.where(sexo == "M", 4.6, 4.0)
    ajuste_idade = np.clip(idade_anos / 10, 0, 1.2)
    peso_kg = rng.normal(loc=base_peso + 0.7 * ajuste_idade, scale=0.7, size=n)
    peso_kg = np.clip(peso_kg, 2.0, 9.5).round(1)

    # Energia (0–100): tende a cair com idade
    energia = (rng.normal(loc=75 - (idade_anos * 2.8), scale=12, size=n)).round(0)
    energia = np.clip(energia, 5, 100).astype(int)

    # Vacinado (exemplo)
    prob_vacinado = np.clip(0.85 - (idade_anos * 0.02), 0.55, 0.90)
    vacinado = rng.random(n) < prob_vacinado

    # Castrado (exemplo)
    prob_castrado = np.clip(0.75 + (idade_anos * 0.01), 0.70, 0.92)
    castrado = rng.random(n) < prob_castrado

    # Visitas ao vet (último ano)
    visitas_vet = rng.poisson(lam=1.0 + (idade_anos * 0.10) + (vacinado.astype(int) * 0.25), size=n)
    visitas_vet = np.clip(visitas_vet, 0, 8)

    # Gosta de caixa 😹
    gosta_caixa = rng.choice([True, False], size=n, p=[0.82, 0.18])

    df = pd.DataFrame({
        "id_gato": [f"G{str(i+1).zfill(5)}" for i in range(n)],
        "id_tutor_hash": id_tutor_hash,
        "data_coleta": pd.to_datetime(data_coleta),

        "sexo": sexo,
        "idade_anos": idade_anos,
        "peso_kg": peso_kg,
        "energia_0a100": energia,
        "vacinado": vacinado,
        "castrado": castrado,
        "pelagem": rng.choice(pelagens, size=n, p=[0.62, 0.26, 0.12]),
        "cor": rng.choice(cores, size=n),
        "temperamento": rng.choice(temperamentos, size=n),
        "alimentacao": rng.choice(alimentacao, size=n, p=[0.55, 0.15, 0.30]),
        "visitas_vet_ult_ano": visitas_vet,
        "gosta_caixa": gosta_caixa
    })

    # Score saúde sintético (somente didático)
    score_saude = (80
                   - (df["idade_anos"] * 2)
                   - (df["visitas_vet_ult_ano"] * 1.5)
                   + (df["energia_0a100"] * 0.15)
                   + (df["vacinado"].astype(int) * 2))
    df["score_saude_0a100"] = np.clip(score_saude.round(0), 10, 100).astype(int)

    return df


# ====== GERAR SMALL E BIG ======
dataset_small = gerar_dataset_gatos(n=1000, seed=42)
dataset_big   = gerar_dataset_gatos(n=10000, seed=42)

print("✅ dataset_small:", dataset_small.shape)
print("✅ dataset_big:", dataset_big.shape)

display(dataset_small.head(5))


# ====== SALVAR NO GOOGLE DRIVE ======
from google.colab import drive
drive.mount("/content/drive")

pasta = "/content/drive/MyDrive/cafe_dados_gatos/"

dataset_small.to_csv(pasta + "dataset_small_gatos.csv", index=False)
dataset_big.to_csv(pasta + "dataset_big_gatos.csv", index=False)

print("✅ Salvos em:", pasta)
print("✅ Arquivos:")
print(" - dataset_small_gatos.csv")
print(" - dataset_big_gatos.csv")


✅ dataset_small: (1000, 16)
✅ dataset_big: (10000, 16)


,id_gato,id_tutor_hash,data_coleta,sexo,idade_anos,peso_kg,energia_0a100,vacinado,castrado,pelagem,cor,temperamento,alimentacao,visitas_vet_ult_ano,gosta_caixa,score_saude_0a100
0,G00001,T9736282219,2025-03-30,M,7,4.4,54,True,True,longa,cinza,caçador,ração seca,1,True,75
1,G00002,T5239865855,2025-05-31,M,6,4.4,63,True,False,longa,laranja,arisco,ração úmida,0,True,79
2,G00003,T9038090091,2025-01-28,M,9,5.2,43,True,True,média,tigrado,arisco,ração seca,3,False,66
3,G00004,T8494103764,2025-10-17,M,9,4.9,57,True,True,curta,cinza,caçador,ração seca,2,True,70
4,G00005,T8326183461,2025-08-26,M,18,4.6,29,True,True,média,tricolor,carinhoso,ração seca,1,True,49


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Salvos em: /content/drive/MyDrive/cafe_dados_gatos/
✅ Arquivos:
 - dataset_small_gatos.csv
 - dataset_big_gatos.csv


# Nova seção